In [139]:
# Environment
from dotenv import load_dotenv

# LangChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# Local Ollama
from langchain_ollama import ChatOllama, OllamaEmbeddings

# Data sources
from data_source.wikipedia import WikipediaSource
from data_source.web_search import WebSearchSource

# Core RAG pipeline
from src.pipeline.indexing_rag import build_vectorstore
from src.pipeline.retrieval_rag import (
    create_retriever,
    retrieve_documents,
)

# Query processing
from src.query_translation.multi_query import (
    create_multi_query_retrieval_chain,
)

# Reranking
from src.Reranking.reranking import CrossEncoderReranker

# RAPTOR
from src.advanced_indexing.raptor import RaptorIndexer

# Self-RAG
from src.advanced_RAG.Self_RAG.self_rag import SelfRAG

# Long context
from src.advanced_RAG.Long_Context.long_context import LongContext

# Memory
from src.memory.memory import ConversationMemory

# Evaluation
from src.evaluation.rag_evaluation import RAGEvaluator

In [140]:
# Environment

load_dotenv()

# Local LLM

llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
    base_url="http://127.0.0.1:11434",
)

# Local embeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest",
    base_url="http://127.0.0.1:11434",
)

In [141]:
# Final generation prompt

generation_prompt = ChatPromptTemplate.from_template("""
You are a careful RAG assistant.

Answer the question using ONLY the provided evidence.

Evidence:
{context}

Question:
{question}

Rules:
- Do not invent facts.
- Do not use unsupported information.
- If the evidence is insufficient, say that the information
  is not available in the retrieved evidence.
- Prefer the most recent and directly relevant evidence.
- Give a clear and concise answer.

Answer:
""")

generation_chain = generation_prompt | llm | StrOutputParser()

In [142]:
# Conversation memory

conversation_memory = ConversationMemory()

# Evaluator

evaluator = RAGEvaluator(
    model="llama3:latest",
    temperature=0,
)

In [143]:
# External data sources

wikipedia = WikipediaSource(
    top_k=5,
)

web_search = WebSearchSource(
    top_k=5,
)

In [144]:
# Ollama answerability prompt

answerability_prompt = ChatPromptTemplate.from_template("""
Determine whether you can answer the question reliably using
your existing knowledge.

Important:
- If the question requires current, live, latest, today's,
  recent, or time-sensitive information, return NOT_ANSWERABLE.
- If you are uncertain, return NOT_ANSWERABLE.
- Return only one label.

ANSWERABLE
NOT_ANSWERABLE

Question:
{question}

Decision:
""")

answerability_chain = answerability_prompt | llm | StrOutputParser()

In [145]:
# User query

query = input("Ask a question: ")

In [146]:
# Local Ollama first

decision = (
    answerability_chain.invoke(
        {
            "question": query,
        }
    )
    .strip()
    .upper()
)

if "NOT_ANSWERABLE" in decision:
    decision = "NOT_ANSWERABLE"
elif "ANSWERABLE" in decision:
    decision = "ANSWERABLE"
else:
    decision = "NOT_ANSWERABLE"

print("Ollama decision:", decision)

Ollama decision: NOT_ANSWERABLE


In [147]:
# Direct Ollama answer

answer = None

if decision == "ANSWERABLE":
    response = llm.invoke(query)
    answer = response.content

    print("\nFinal Answer:")
    print(answer)

In [148]:
decision == "NOT_ANSWERABLE"

True

In [149]:
# Detect current/time-sensitive queries


def is_current_query(query: str) -> bool:
    keywords = [
        "today",
        "current",
        "latest",
        "now",
        "live",
        "recent",
        "yesterday",
        "tomorrow",
    ]

    query_lower = query.lower()

    return any(keyword in query_lower for keyword in keywords)

In [150]:
# External retrieval

if decision == "NOT_ANSWERABLE":

    if is_current_query(query):

        # Current / time-sensitive information
        documents = web_search.retrieve(query)

        print("Source: Tavily Web Search")

        print(
            "Web documents:",
            len(documents),
        )

    else:

        # Stable external knowledge
        wikipedia_documents = wikipedia.retrieve(query)

        web_documents = web_search.retrieve(query)

        documents = wikipedia_documents + web_documents

        print(
            "Wikipedia:",
            len(wikipedia_documents),
        )

        print(
            "Web:",
            len(web_documents),
        )

        print(
            "Total:",
            len(documents),
        )

Source: Tavily Web Search
Web documents: 5


In [151]:
# Deduplicate documents

unique_documents = {}

for document in documents:
    url = document.metadata.get("url")

    if url:
        key = url
    else:
        key = document.metadata.get("title", "") + document.page_content[:200]

    if key not in unique_documents:
        unique_documents[key] = document

documents = list(unique_documents.values())

print("Unique documents:", len(documents))

Unique documents: 5


In [152]:
# RAPTOR indexing

if decision == "NOT_ANSWERABLE":

    raptor = RaptorIndexer(
        llm=llm,
        embeddings=embeddings,
        n_clusters=3,
    )

    raptor.build_tree(
        documents=documents,
    )

    raptor_results = raptor.retrieve(
        query=query,
        k=3,
    )

    raptor_leaf = raptor_results.get(
        "leaf",
        [],
    )

    raptor_clusters = raptor_results.get(
        "clusters",
        [],
    )

    print(
        "RAPTOR leaf:",
        len(raptor_leaf),
    )

    print(
        "RAPTOR clusters:",
        len(raptor_clusters),
    )

RAPTOR leaf: 3
RAPTOR clusters: 3


In [153]:
# Build vector store for stable external knowledge

if decision == "NOT_ANSWERABLE" and not is_current_query(query):
    vectorstore = build_vectorstore(
        documents=documents,
        embedding_model=embeddings,
        batch_size=32,
    )

    print("Vector store created.")

In [154]:
# Create retriever

if decision == "NOT_ANSWERABLE" and not is_current_query(query):
    retriever = create_retriever(
        vectorstore=vectorstore,
        k=5,
    )

In [155]:
# Create Multi-Query retriever

if decision == "NOT_ANSWERABLE" and not is_current_query(query):
    multi_query_retriever = create_multi_query_retrieval_chain(
        retriever=retriever,
        llm=llm,
    )

In [156]:
# Multi-Query retrieval

if decision == "NOT_ANSWERABLE" and not is_current_query(query):
    retrieved_documents = multi_query_retriever.invoke(query)

    print(
        "Retrieved documents:",
        len(retrieved_documents),
    )

In [157]:
# Reranking

if decision == "NOT_ANSWERABLE" and is_current_query(query):
    reranker = CrossEncoderReranker(
        top_k=5,
    )

    reranked_documents = reranker.rerank_documents(
        query=query,
        documents=documents,
    )

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [158]:
# Reranking stable RAG results

if decision == "NOT_ANSWERABLE" and not is_current_query(query):
    reranked_documents = reranker.rerank_documents(
        query=query,
        documents=retrieved_documents,
    )

In [159]:
# Build final evidence

if decision == "NOT_ANSWERABLE":

    reranked_context = "\n\n".join(
        document.page_content for document in reranked_documents
    )

    raptor_context = "\n\n".join(str(item) for item in (raptor_leaf + raptor_clusters))

    final_context_parts = [
        reranked_context,
        raptor_context,
        self_rag_answer,
    ]

    context = "\n\n".join(part for part in final_context_parts if part)

    print(
        "Final context length:",
        len(context),
    )

Final context length: 11015


In [160]:
# Long-context processing

if decision == "NOT_ANSWERABLE":

    long_context = LongContext(
        model="llama3:latest",
    )

    long_context_result = long_context.run(
        documents=reranked_documents,
        query=query,
        compress=True,
    )

    long_context_answer = long_context_result.get(
        "answer",
        "",
    )

    if long_context_answer:
        context += "\n\n" + long_context_answer

In [161]:
# Final RAG generation

if decision == "NOT_ANSWERABLE":

    answer = generation_chain.invoke(
        {
            "context": context,
            "question": query,
        }
    )

    print("\nFinal RAG Answer:")
    print(answer)


Final RAG Answer:
According to the provided context, the NEPSE Index closed at 2,599.58 points as of August 24, 2026.


In [162]:
# Store conversation memory

conversation_memory.add_message(HumanMessage(content=query))

conversation_memory.add_message(AIMessage(content=answer))

In [163]:
# Evaluation

if decision == "ANSWERABLE":

    answer_relevance = evaluator.evaluate_answer_relevance(
        question=query,
        answer=answer,
    )

    print(
        "Answer Relevance:",
        answer_relevance,
    )

else:

    context_relevance = evaluator.evaluate_context_relevance(
        question=query,
        context=context,
    )

    faithfulness = evaluator.evaluate_faithfulness(
        context=context,
        answer=answer,
    )

    answer_relevance = evaluator.evaluate_answer_relevance(
        question=query,
        answer=answer,
    )

    print(
        "Context Relevance:",
        context_relevance,
    )

    print(
        "Faithfulness:",
        faithfulness,
    )

    print(
        "Answer Relevance:",
        answer_relevance,
    )

Context Relevance: RELEVANT

The context provides information about the NEPSE Index, including its value as of August 24, 2026, which is 2,599.58 points. This information is directly relevant to answering the question about the NEPSE index today.
Faithfulness: **FAITHFUL**

The answer is fully supported by the provided context. The context states: "NEPSE Index: 2599.58 (-19.14 pts, -0.73%)". This information is explicitly mentioned in the context, and there is no ambiguity or uncertainty that would suggest the answer is not faithful to the provided context.
Answer Relevance: RELEVANT


In [164]:
# Final output

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(answer)
print("=" * 60)


FINAL ANSWER
According to the provided context, the NEPSE Index closed at 2,599.58 points as of August 24, 2026.
